In [3]:
from __future__ import annotations

import logging
from pathlib import Path

import numpy as np
import pandas as pd

from credit_risk.utils.config import read_config, create_path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)-20s  %(message)s",
)

logger = logging.getLogger("v2_data_overview")

In [4]:
from pathlib import Path
import os

if "project_path" not in globals():
    project_path = Path.cwd().parent
    os.chdir(project_path)

print("Project path:", project_path)

Project path: c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk


In [ ]:
config = read_config(project_path)

In [7]:
approach = config["parameters"]["modelling_approach"]
modelling_config = config["parameters"]["modelling"]

train_vintages = modelling_config["vintages_train"]
validation_vintages = modelling_config["vintages_test"]
oot_vintages = modelling_config["vintages_oot"]

target = config["parameters"]["target"]["name"]

logger.info("Modelling approach: %s", approach)
logger.info("Train vintages: %s", train_vintages)
logger.info("Validation vintages: %s", validation_vintages)
logger.info("OOT vintages: %s", oot_vintages)
logger.info("Target: %s", target)

2026-08-23 23:55:26,335  INFO      v2_data_overview      Modelling approach: behavioral
2026-08-23 23:55:26,337  INFO      v2_data_overview      Train vintages: [2015, 2016, 2017, 2018]
2026-08-23 23:55:26,338  INFO      v2_data_overview      Validation vintages: [2019, 2020]
2026-08-23 23:55:26,339  INFO      v2_data_overview      OOT vintages: [2021, 2022]
2026-08-23 23:55:26,340  INFO      v2_data_overview      Target: future_90dpd_12m


In [21]:
train_path = create_path(
    config["catalog"]["base"],
    config["catalog"],
    "train_df",
    approach,

)
test_path = create_path(
    config["catalog"]["base"],
    config["catalog"],
    "validation_df",
    approach,

)
oot_path = create_path(
    config["catalog"]["base"],
    config["catalog"],
    "oot_df",
    approach,
    must_exist=False,
)
train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)
oot_df= pd.read_parquet(oot_path)
df = pd.concat([train_df,test_df,oot_df],ignore_index=True)

logger.info(
    "Modelling dataset loaded: rows=%s columns=%s",
    f"{len(df):,}",
    len(df.columns),
)

df.head()

2026-08-24 00:02:38,693  INFO      v2_data_overview      Modelling dataset loaded: rows=1,508,462 columns=46


,loan_id,period,current_actual_upb,current_interest_rate,loan_age,remaining_months_to_legal_maturity,estimated_ltv,current_loan_delinquency_status,ddlpi,zero_balance_code,...,ever_30dpd_to_date,ever_60dpd_to_date,delinquency_months_to_date,months_since_last_delinquency,upb_change_from_origination,upb_pct_change_from_origination,rate_change_from_origination,observation_age,future_90dpd_12m,vintage
0,F15Q10000025,2015-06,409000.0,2.5,3,177,999,00,NaT,<NA>,...,0,0,0,NaN,-8000.0,-0.019185,0.0,3,0,2015
1,F15Q10000031,2015-06,109000.0,3.625,3,357,999,00,NaT,<NA>,...,0,0,0,NaN,-1000.0,-0.009091,0.0,3,0,2015
2,F15Q10000084,2015-06,294000.0,3.375,3,177,999,00,NaT,<NA>,...,0,0,0,NaN,-3000.0,-0.010101,0.0,3,0,2015
3,F15Q10000087,2015-05,214000.0,4.0,3,357,999,00,NaT,<NA>,...,0,0,0,NaN,-1000.0,-0.004651,0.0,3,0,2015
4,F15Q10000157,2015-06,178000.0,3.75,3,177,999,00,NaT,<NA>,...,0,0,0,NaN,-2000.0,-0.011111,0.0,3,0,2015


In [22]:
overview = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "unique_loans",
            "vintages",
            "observation_ages",
        ],
        "value": [
            len(df),
            len(df.columns),
            df["loan_id"].nunique() if "loan_id" in df.columns else np.nan,
            df["vintage"].nunique() if "vintage" in df.columns else np.nan,
            (
                df["observation_age"].nunique()
                if "observation_age" in df.columns
                else np.nan
            ),
        ],
    }
)

overview

,metric,value
0,rows,1508462
1,columns,46
2,unique_loans,393908
3,vintages,8
4,observation_ages,4


In [37]:
vintage_summary = (
    df.groupby("vintage")
    .agg(
        population=("vintage", "size"),
        loans=(
            ("loan_id", "nunique") if "loan_id" in df.columns else ("vintage", "size")
        ),
        events=(target, "sum"),
        event_rate=(target, "mean"),
    )
    .reset_index()
    .sort_values("vintage")
)

vintage_summary["population_share"] = (
    vintage_summary["population"] / vintage_summary["population"].sum()
)

print(vintage_summary)

   vintage  population  loans  events  event_rate  population_share
0     2015      187625  48266     543    0.002894          0.124382
1     2016      192551  49308     841    0.004368          0.127647
2     2017      191312  49099     844    0.004412          0.126826
3     2018      187336  49453    2490    0.013292          0.124190
4     2019      178958  49480    6209    0.034695          0.118636
5     2020      183537  49359    1305    0.007110          0.121672
6     2021      193330  49549    1030    0.005328          0.128164
7     2022      193813  49394    2077    0.010717          0.128484


In [24]:
observation_age_summary = (
    df.groupby("observation_age")
    .agg(
        population=("observation_age", "size"),
        loans=("loan_id", "nunique"),
        events=(target, "sum"),
        event_rate=(target, "mean"),
    )
    .reset_index()
    .sort_values("observation_age")
)

observation_age_summary["population_share"] = (
    observation_age_summary["population"] / observation_age_summary["population"].sum()
)

observation_age_summary

,observation_age,population,loans,events,event_rate,population_share
0,3,392538,392538,3543,0.009026,0.260224
1,6,385026,385026,3878,0.010072,0.255244
2,9,372633,372633,3950,0.010600,0.247028
3,12,358265,358265,3968,0.011076,0.237503


In [25]:
vintage_age_summary = (
    df.groupby(["vintage", "observation_age"])
    .agg(
        population=("observation_age", "size"),
        loans=("loan_id", "nunique"),
        events=(target, "sum"),
        event_rate=(target, "mean"),
    )
    .reset_index()
    .sort_values(["vintage", "observation_age"])
)

vintage_age_event_rate = vintage_age_summary.pivot(
    index="vintage",
    columns="observation_age",
    values="event_rate",
)

vintage_age_event_rate

observation_age,3,6,9,12
vintage,,,,
2015,0.001994,0.002606,0.002901,0.004146
2016,0.002560,0.004197,0.005358,0.005431
2017,0.004533,0.004435,0.004257,0.004418
2018,0.003368,0.008023,0.016552,0.026575
2019,0.034229,0.039054,0.035271,0.029278
2020,0.011513,0.006522,0.005500,0.004323
2021,0.004747,0.005428,0.005409,0.005751
2022,0.009014,0.010698,0.011311,0.011903


In [26]:
target_checks = {
    "target_missing": int(df[target].isna().sum()),
    "target_unique_values": int(df[target].nunique()),
    "target_min": df[target].min(),
    "target_max": df[target].max(),
    "event_count": int(df[target].sum()),
    "event_rate": float(df[target].mean()),
}

target_checks

{'target_missing': 0,
 'target_unique_values': 2,
 'target_min': np.int8(0),
 'target_max': np.int8(1),
 'event_count': 15339,
 'event_rate': 0.01016863533851035}

In [27]:
loan_observation_counts = df.groupby("loan_id").size().rename("observation_count")

loan_observation_counts.describe()

count    393908.000000
mean          3.829478
std           0.581979
min           1.000000
25%           4.000000
50%           4.000000
75%           4.000000
max           4.000000
Name: observation_count, dtype: float64

In [28]:
loan_observation_counts.value_counts().sort_index()

observation_count
1      8605
2     13035
3     15285
4    356983
Name: count, dtype: int64

In [29]:
configured_vintages = set(train_vintages) | set(validation_vintages) | set(oot_vintages)

available_vintages = set(df["vintage"].dropna().unique())

missing_configured_vintages = sorted(configured_vintages - available_vintages)

unexpected_vintages = sorted(available_vintages - configured_vintages)

print("Missing configured vintages:", missing_configured_vintages)
print("Unexpected vintages:", unexpected_vintages)
if missing_configured_vintages:
    raise ValueError(
        "Configured vintages missing from modelling dataset: "
        + ", ".join(map(str, missing_configured_vintages))
    )

Missing configured vintages: []
Unexpected vintages: []


In [31]:
split_summary = pd.DataFrame(
    {
        "split": [
            "train",
            "validation",
            "oot",
        ],
        "vintages": [
            ", ".join(map(str, train_vintages)),
            ", ".join(map(str, validation_vintages)),
            ", ".join(map(str, oot_vintages)),
        ],
        "rows": [
            len(train_df),
            len(test_df),
            len(oot_df),
        ],
        "loans": [
            train_df["loan_id"].nunique(),
            test_df["loan_id"].nunique(),
            oot_df["loan_id"].nunique(),
        ],
        "events": [
            train_df[target].sum(),
            test_df[target].sum(),
            oot_df[target].sum(),
        ],
        "event_rate": [
            train_df[target].mean(),
            test_df[target].mean(),
            oot_df[target].mean(),
        ],
    }
)

split_summary

,split,vintages,rows,loans,events,event_rate
0,train,"2015, 2016, 2017, 2018",758824,196126,4718,0.006218
1,validation,"2019, 2020",362495,98839,7514,0.020729
2,oot,"2021, 2022",387143,98943,3107,0.008025


In [34]:
train_loans = set(train_df["loan_id"])
validation_loans = set(test_df["loan_id"])
oot_loans = set(oot_df["loan_id"])

print(
    "Train / validation overlap:",
    len(train_loans.intersection(validation_loans)),
)

print(
    "Train / OOT overlap:",
    len(train_loans.intersection(oot_loans)),
)

print(
    "Validation / OOT overlap:",
    len(validation_loans.intersection(oot_loans)),
)

if train_loans.intersection(validation_loans):
    raise ValueError("Train/validation loan leakage detected.")

if train_loans.intersection(oot_loans):
    raise ValueError("Train/OOT loan leakage detected.")

if validation_loans.intersection(oot_loans):
    raise ValueError("Validation/OOT loan leakage detected.")

Train / validation overlap: 0
Train / OOT overlap: 0
Validation / OOT overlap: 0


In [35]:
print("=== V2 DATA OVERVIEW ===")
print(f"Rows: {len(df):,}")
print(f"Loans: {df['loan_id'].nunique():,}")
print(f"Overall event rate: {df[target].mean():.4%}")
print(f"Train event rate: {train_df[target].mean():.4%}")
print(f"Validation event rate: {test_df[target].mean():.4%}")
print(f"OOT event rate: {oot_df[target].mean():.4%}")

print(
    "Train / validation loan overlap:",
    len(train_loans & validation_loans),
)
print(
    "Train / OOT loan overlap:",
    len(train_loans & oot_loans),
)
print(
    "Validation / OOT loan overlap:",
    len(validation_loans & oot_loans),
)

=== V2 DATA OVERVIEW ===
Rows: 1,508,462
Loans: 393,908
Overall event rate: 1.0169%
Train event rate: 0.6218%
Validation event rate: 2.0729%
OOT event rate: 0.8025%
Train / validation loan overlap: 0
Train / OOT loan overlap: 0
Validation / OOT loan overlap: 0
